This notebook computes the **Percent Trips Connected** metric following the
methodology of Mekuria, Furth & Nixon (2012), applied to the Belfast Local
Government District using the LTS-classified cycling network from the companion
notebook (`belfast_cycle_lts_assessment.ipynb`).

## Methodology Overview

Two points are said to be *connected* at a given LTS level if there exists a
path between them using only links that do not exceed that level of stress,
and the path does not involve undue detour. The detour criterion requires that
the low-stress path length must not exceed the shortest (all-network) path by
more than **25%**, or for short trips, **0.53 km (0.33 miles)**.

**Percent Trips Connected** is the share of total commuting demand (from Census
2021 OD data) that can be served by a connected, low-stress route at each LTS
threshold.

### Data Sources

| Dataset | Source | Spatial unit |
|---|---|---|
| LTS network | `belfast_cycle_lts_assessment.ipynb` output | Edge-level |
| OD commuting flows | PCTNI / Census 2021 ODWP01 | Data Zone (DZ) |
| Data Zone boundaries | NISRA `DZ2021.geojson` | DZ |
| Super Data Zone boundaries | PCTNI `zones_sdz.gpkg` | SDZ |

### Pipeline

1. Load LTS network, DZ/SDZ boundaries, and OD data
2. Build DZ → SDZ lookup via spatial join
3. Aggregate OD flows from DZ to SDZ level
4. Construct routable graph from LTS network
5. Snap SDZ centroids to nearest network node
6. For each LTS threshold (1–4), compute connectivity for all OD pairs
7. Calculate Percent Trips Connected and Percent Nodes Connected
8. Visualise results

## Environment Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from shapely.geometry import Point
from shapely.ops import nearest_points

# Network analysis — igraph is much faster than networkx for large graphs
import igraph as ig
import networkx as nx

from collections import defaultdict
from itertools import combinations

# Project paths
DATA_DIR = Path('../Data')
OUTPUT_DIR = Path('../Figure')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Coordinate reference systems
CRS_WGS84 = 'EPSG:4326'
CRS_IG    = 'EPSG:29903'   # Irish Grid — all spatial ops use this

print('Environment ready.')


Environment ready.


## Load Data

### LTS Network

Load the LTS-classified cycling network produced by the companion notebook.
Each edge has an `lts` column (1–4) and geometry in Irish Grid.

In [3]:
# ── Load the LTS-classified network ──────────────────────────────────────────
# This parquet file is the output of belfast_cycle_lts_assessment.ipynb
# Adjust the filename if your output is named differently
edges = gpd.read_parquet(DATA_DIR / 'LTS_Belfast.parquet')
edges = edges.to_crs(CRS_IG)

print(f'Total edges: {len(edges):,}')
print(f'CRS: {edges.crs}')
print(f'\nLTS distribution:')
print(edges['lts'].value_counts().sort_index())
print(f'\nNaN LTS: {edges["lts"].isna().sum():,}')


Total edges: 23,464
CRS: EPSG:29903

LTS distribution:
lts
1.0    9752
2.0    9886
3.0    1528
4.0    2298
Name: count, dtype: int64

NaN LTS: 0


### Study Area — Belfast LGD Zones

Load SDZ boundaries and filter to Belfast LGD.

In [6]:
# ── SDZ zones ────────────────────────────────────────────────────────────────
sdz_all = gpd.read_file(DATA_DIR / 'zones_sdz.gpkg', engine='pyogrio')
sdz_all = sdz_all.to_crs(CRS_IG)

# Filter to Belfast LGD
belfast_sdz = sdz_all[sdz_all['lgd2014_nm'] == 'Belfast'].copy()
belfast_boundary = belfast_sdz.unary_union

print(f'Belfast SDZ count: {len(belfast_sdz)}')
print(f'Belfast area: {belfast_sdz.geometry.area.sum() / 1e6:.1f} km²')


Belfast SDZ count: 175
Belfast area: 137.7 km²


### OD Commuting Data

Load the [Census 2021 OD flows](https://www.nisra.gov.uk/publications/data-zone-boundaries-gis-format). The PCTNI filtered version retains only
`place_of_work_ind_code == 4` (fixed workplace within NI), with both
origin and destination coded at SDZ level.

In [13]:
# ── OD data ──────────────────────────────────────────────────────────────────
od_raw = pd.read_csv(DATA_DIR / 'od_ni_open_filtered.csv')

print(f'Total OD records: {len(od_raw):,}')
print(f'Total commuters: {od_raw["count"].sum():,}')
print(f'Columns: {list(od_raw.columns)}')
od_raw.head()


Total OD records: 108,656
Total commuters: 545,210
Columns: ['area_of_residence_code', 'workplace_area_code', 'place_of_work_ind_code', 'count']


,area_of_residence_code,workplace_area_code,place_of_work_ind_code,count
0,N21000001,N21000001,4,51
1,N21000001,N21000002,4,3
2,N21000001,N21000003,4,4
3,N21000001,N21000004,4,34
4,N21000001,N21000005,4,28


## OD Data Preparation

The Census 2021 OD commuting flows (ODWP01) are recorded at Super Data Zone
(SDZ) level — the finest geography available in this dataset. Since the OD
codes match `zones_sdz.gpkg` directly, no spatial aggregation is required.
This section filters the OD pairs to those with both origin and destination
within Belfast LGD, and excludes intra-zone trips following Mekuria et al.
(2012) to limit the influence of very short trips for which walking is the
dominant mode.

In [18]:
# ── Identify SDZ code column ────────────────────────────────────────────────
sdz_code_col = [c for c in belfast_sdz.columns if 'sdz' in c.lower() and 'code' in c.lower()]
if not sdz_code_col:
    sdz_code_col = belfast_sdz.columns[0]  # fallback
else:
    sdz_code_col = sdz_code_col[0]
print(f'Using SDZ code column: "{sdz_code_col}"')

# ── Filter OD to Belfast: both origin AND destination within Belfast LGD ────
belfast_sdz_codes = set(belfast_sdz[sdz_code_col].values)

od = od_raw[
    od_raw['area_of_residence_code'].isin(belfast_sdz_codes) &
    od_raw['workplace_area_code'].isin(belfast_sdz_codes)
].copy()

od = od.rename(columns={
    'area_of_residence_code': 'origin_sdz',
    'workplace_area_code': 'dest_sdz',
})

# Exclude intra-zone trips (Mekuria et al. 2012)
od = od[od['origin_sdz'] != od['dest_sdz']].copy()

print(f'Belfast inter-SDZ OD pairs: {len(od):,}')
print(f'Total inter-SDZ commuters: {od["count"].sum():,}')
print(f'Unique origin SDZs: {od["origin_sdz"].nunique()}')
print(f'Unique destination SDZs: {od["dest_sdz"].nunique()}')


Using SDZ code column: "sdz2021_cd"
Belfast inter-SDZ OD pairs: 13,412
Total inter-SDZ commuters: 68,877
Unique origin SDZs: 175
Unique destination SDZs: 175
